# Chapter 10: Natural Language Processing Basics
## Notebook 02 — NLP Classification & NER

This notebook covers **deep learning for text** (CNN/LSTM), **multi-class text classification**, **named entity recognition** with spaCy, **text similarity and clustering**, and common **NLP pitfalls**.

### What you'll learn

| Topic | Section |
|-------|--------|
| Deep learning for NLP (RNNs, LSTMs) | §1–2 |
| Multi-class text classification | §3 |
| Named Entity Recognition (NER) | §4–5 |
| Text similarity and clustering | §6 |
| Full pipeline & pitfalls | §7–8 |

**Estimated time:** 2.5–3 hours

---
*Generated by Berta AI*

---
## 1. Deep Learning for NLP: Overview

Traditional methods (TF-IDF + classifier) ignore **word order**. **RNNs** and **LSTMs** process sequences step-by-step and can capture context. **CNNs** on top of embeddings can also work well for classification. We'll use **TF-IDF + Logistic Regression** for a quick multi-class baseline, then discuss LSTM architecture (see diagram in `assets/diagrams/lstm_architecture.mermaid`).

In [ ]:
import sys
import os
sys.path.insert(0, os.path.join(os.getcwd(), '..', 'scripts'))

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, f1_score

np.random.seed(42)
print("Imports OK.")

---
## 2. Text Classification with Embeddings (and TF-IDF baseline)

We build a **multi-class classifier** using TF-IDF + Logistic Regression. For LSTM you would: embed each token, pass through LSTM, pool (e.g. last hidden state), then dense → softmax. Here we keep the pipeline simple and runnable without a large embedding file.

In [ ]:
from nlp_models import TextClassifier

DATA_DIR = os.path.join('..', 'datasets')
news_path = os.path.join(DATA_DIR, 'news_categories.csv')

if os.path.exists(news_path):
    df = pd.read_csv(news_path)
    texts = df['text'].fillna('').tolist()
    categories = df['category'].astype(str).tolist()
    from sklearn.preprocessing import LabelEncoder
    le = LabelEncoder()
    labels = le.fit_transform(categories)
    num_classes = len(le.classes_)
    print("Classes:", le.classes_)
else:
    texts = ["Tech news about AI.", "Sports score today.", "Politics and elections."] * 5
    labels = [0, 1, 2] * 5
    num_classes = 3
    le = None

X_train, X_test, y_train, y_test = train_test_split(texts, labels, test_size=0.25, random_state=42)

clf = TextClassifier(num_classes=num_classes, model_type='logistic')
clf.train(X_train, y_train)
preds = [clf.predict(t)[0] for t in X_test]
print("F1 (macro):", f1_score(y_test, preds, average='macro'))
print(classification_report(y_test, preds))
print(confusion_matrix(y_test, preds))

---
## 3. Practical Project: Multi-class Text Classification

We've already trained a multi-class model above. Key steps: load data → preprocess → vectorize (TF-IDF) → train classifier → evaluate (precision, recall, F1, confusion matrix). For **LSTM**: you'd need tokenization, padding, embedding matrix, then LSTM + dense. The same `TextClassifier` API could be extended to accept `model_type='lstm'` and use Keras/PyTorch under the hood.

---
## 4. Named Entity Recognition (NER) Introduction

**NER** identifies spans of text that are persons (PERSON), organizations (ORG), locations (GPE, LOC), dates (DATE), etc. **spaCy** provides a pre-trained NER pipeline.

In [ ]:
from nlp_models import NERModel

ner = NERModel()
text = "John Smith lives in New York City. Apple Inc. was founded by Steve Jobs."
entities = ner.extract_entities(text)
print("Entities:")
for e in entities:
    print(f"  {e['text']} -> {e['label']}")

---
## 5. Building a Custom NER Tagger (Concept)

A **BIO tagging scheme** (Begin, Inside, Outside) labels each token. A **BiLSTM-CRF** is a common choice: bidirectional LSTM for context + CRF for valid label sequences. Implementation is beyond this notebook; see Chapter 11 for transformer-based NER.

---
## 6. Text Similarity and Clustering

Using TF-IDF vectors we can compute **cosine similarity** between documents and **cluster** them (e.g. K-Means).

In [ ]:
from nlp_models import TextSimilarity

sim = TextSimilarity(method='tfidf')
documents = [
    "Machine learning is a subset of AI.",
    "Artificial intelligence includes ML and deep learning.",
    "The weather today is sunny."
]
sim.fit(documents)
print("Similarity doc0 vs doc1:", sim.similarity(documents[0], documents[1]))
print("Similarity doc0 vs doc2:", sim.similarity(documents[0], documents[2]))
labels_cluster = sim.cluster_texts(documents, n_clusters=2)
print("Cluster labels:", labels_cluster)

---
## 7. Project Checkpoint: Full NLP Pipeline

Combine: **sentiment** (SentimentAnalyzer) + **NER** (NERModel) + **classification** (TextClassifier). Load text → run sentiment → run NER → run category prediction. Build one function that runs all three and returns a dict.

In [ ]:
from nlp_models import SentimentAnalyzer, NERModel, TextClassifier

def full_pipeline(text: str, sentiment_model, ner_model, classifier):
    out = {}
    if sentiment_model:
        pred, conf = sentiment_model.predict(text)
        out['sentiment'] = pred
        out['sentiment_confidence'] = conf
    if ner_model:
        out['entities'] = ner_model.extract_entities(text)
    if classifier:
        pred, proba = classifier.predict(text)
        out['category'] = pred
        out['category_proba'] = proba
    return out

# Example: need trained sentiment and classifier; NER is ready.
ner = NERModel()
sample = "Apple announced a new product in Cupertino. Investors are excited."
result = full_pipeline(sample, None, ner, None)
print("Pipeline output (NER only):", result)

---
## 8. Common NLP Pitfalls

- **Overfitting**: Use regularization, more data, or simpler models (e.g. TF-IDF + LR).
- **Data imbalance**: Balance classes (resampling) or use class_weight in classifiers.
- **Out-of-vocabulary (OOV)**: Use unknown token, subword tokenization, or pre-trained embeddings with OOV handling.
- **Efficiency**: For large corpora use sparse matrices and batch inference; consider model size and hardware.

Next: **Notebook 03** — attention, seq2seq, transfer learning, production.

---
*Generated by Berta AI*